# Exercise 8 — Predictive Maintenance & Control Decisions

In manufacturing, control system behavior tells us more than just *is it tracking?*
It tells us about **equipment health**, **production quality risk**, and **when to act**.

This exercise connects response metrics to operational decisions.

## Section 1 — Setup & Simulation Engine

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pdm_helpers import compute_metrics, compute_kpis, recommend_action, print_dashboard


def simulate(
    wn=2.0, zeta=0.4, K_plant=1.0, J=0.5,
    Kp=5.0, Ki=2.0, Kd=0.5,
    u_min=0.0, u_max=10.0,
    sensor_bias=0.0, actuator_scale=1.0,
    extra_friction=0.0, dist_load=0.0,
    dt=0.005, t_end=30.0, target=1.0,
):
    """Run closed-loop simulation and return results dict."""
    t = np.arange(0, t_end, dt)
    n = len(t)
    ref = np.full(n, target)
    dist = np.zeros(n)
    dist[t >= 10.0] = dist_load

    y = np.zeros(n)
    dy = np.zeros(n)
    u = np.zeros(n)
    integral_e = 0.0
    prev_e = 0.0

    for i in range(1, n):
        y_meas = y[i - 1] + sensor_bias
        e = ref[i - 1] - y_meas
        integral_e += e * dt
        de = (e - prev_e) / dt
        prev_e = e

        u_raw = Kp * e + Ki * integral_e + Kd * de
        u_cmd = np.clip(u_raw, u_min, u_max)
        if u_cmd != u_raw:
            integral_e -= e * dt

        u_applied = u_cmd * actuator_scale
        u[i] = u_applied

        ddy = (
            wn**2 * K_plant * u_applied
            - 2 * zeta * wn * dy[i - 1]
            - wn**2 * y[i - 1]
            - dist[i - 1] / J
            - extra_friction * dy[i - 1] / J
        )
        dy[i] = dy[i - 1] + ddy * dt
        y[i] = max(y[i - 1] + dy[i] * dt, 0.0)

    return dict(t=t, y=y, ref=ref, u=u)

## Section 2 — Healthy Baseline

Run the nominal system and examine its metrics.

In [ ]:
# ---- Healthy baseline ----
res = simulate()
metrics = compute_metrics(res["t"], res["y"], res["ref"], res["u"], u_max=10.0)
kpis = compute_kpis(metrics)
rec = recommend_action(metrics, kpis)
print_dashboard(metrics, kpis, rec)

In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Output Tracking", "Control Effort"),
    vertical_spacing=0.10,
)
fig.add_trace(
    go.Scatter(x=res["t"], y=res["ref"], mode="lines", name="Target",
               line=dict(color="black", dash="dash", width=1.5)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=res["t"], y=res["y"], mode="lines", name="Output",
               line=dict(color="#636EFA", width=2)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=res["t"], y=res["u"], mode="lines", name="Control",
               line=dict(color="#EF553B", width=1.5)),
    row=2, col=1,
)
fig.update_yaxes(title_text="Speed [m/s]", row=1, col=1)
fig.update_yaxes(title_text="Voltage [V]", row=2, col=1)
fig.update_xaxes(title_text="Time [s]", row=2, col=1)
fig.update_layout(
    template="plotly_white", height=500, title_text="Healthy Baseline",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(t=80, b=40),
)
fig.show()

## Section 3 — Degraded Scenarios

Now we run several degraded conditions and compare the dashboards.

### Scenario A: Worn Bearings (increased friction)

In [ ]:
res_a = simulate(extra_friction=3.0)
m_a = compute_metrics(res_a["t"], res_a["y"], res_a["ref"], res_a["u"], u_max=10.0)
k_a = compute_kpis(m_a)
r_a = recommend_action(m_a, k_a)
print("SCENARIO A: Worn Bearings")
print_dashboard(m_a, k_a, r_a)

### Scenario B: Aggressive Tuning (Kp too high)

In [ ]:
res_b = simulate(Kp=15.0, Ki=5.0, Kd=0.2)
m_b = compute_metrics(res_b["t"], res_b["y"], res_b["ref"], res_b["u"], u_max=10.0)
k_b = compute_kpis(m_b)
r_b = recommend_action(m_b, k_b)
print("SCENARIO B: Aggressive Tuning")
print_dashboard(m_b, k_b, r_b)

### Scenario C: Weak Actuator + Heavy Load

In [ ]:
res_c = simulate(actuator_scale=0.5, dist_load=2.5)
m_c = compute_metrics(res_c["t"], res_c["y"], res_c["ref"], res_c["u"], u_max=10.0)
k_c = compute_kpis(m_c)
r_c = recommend_action(m_c, k_c)
print("SCENARIO C: Weak Actuator + Heavy Load")
print_dashboard(m_c, k_c, r_c)

## Section 4 — Visual Comparison

In [ ]:
all_scenarios = {
    "Healthy": res,
    "Worn bearings": res_a,
    "Aggressive tuning": res_b,
    "Weak + loaded": res_c,
}
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Output Tracking", "Control Effort"),
    vertical_spacing=0.10,
)

for (name, r), color in zip(all_scenarios.items(), colors):
    fig.add_trace(
        go.Scatter(x=r["t"], y=r["y"], mode="lines",
                   name=name, line=dict(color=color, width=2)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=r["t"], y=r["u"], mode="lines",
                   name=name, line=dict(color=color, width=1.5),
                   showlegend=False),
        row=2, col=1,
    )

fig.add_trace(
    go.Scatter(x=res["t"], y=res["ref"], mode="lines", name="Target",
               line=dict(color="black", dash="dash", width=1.5)),
    row=1, col=1,
)
fig.update_yaxes(title_text="Speed [m/s]", row=1, col=1)
fig.update_yaxes(title_text="Voltage [V]", row=2, col=1)
fig.update_xaxes(title_text="Time [s]", row=2, col=1)
fig.update_layout(
    template="plotly_white", height=600, title_text="Scenario Comparison",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(t=80, b=40),
)
fig.show()

## Section 5 — Metrics Interpretation Guide

| Metric | What It Measures | Manufacturing Meaning |
|---|---|---|
| Overshoot | Peak above target | Product damage risk, over-speed, over-temperature |
| Settling time | Time to reach and stay on target | Startup waste, lost throughput during transitions |
| Steady-state error | Persistent offset from target | Consistent quality drift, calibration problem |
| Oscillation count | Repeated over/undershoot | Mechanical stress, fatigue, inconsistent product |
| Time outside band | Duration outside tolerance | Scrap/rework rate, SPC violations |
| Saturation exposure | Actuator at its limits | Energy waste, actuator wear, reduced headroom for disturbances |

## Section 6 — KPI Interpretation

| KPI | What Drives It | Plant-Floor Consequence |
|---|---|---|
| Throughput Risk | Slow settling, time off-target | Reduced output rate, startup waste |
| Quality Risk | Overshoot, oscillation | Scrap, rework, customer complaints |
| Energy Penalty | Saturation, high error | Wasted energy, higher utility costs |
| Maintenance Stress | Oscillation, saturation | Accelerated wear on actuators, bearings, belts |

## Section 7 — Student Challenge

You are the process engineer for a conveyor line. Run the three scenarios below,
examine the dashboard for each, and fill in the decision table.

| Scenario | Your Recommendation | Justification (cite 1-2 metrics + 1 KPI) |
|---|---|---|
| `simulate(extra_friction=1.5)` | ___ | ___ |
| `simulate(Kp=12.0, Kd=0.1)` | ___ | ___ |
| `simulate(actuator_scale=0.7, dist_load=1.5)` | ___ | ___ |

**Valid recommendations:** continue running, inspect soon, reduce load, retune controller, schedule maintenance shutdown

In [ ]:
# ---- Student challenge ----
# Run each scenario, compute metrics/KPIs, and print the dashboard
# Example:
# res_x = simulate(extra_friction=1.5)
# m_x = compute_metrics(res_x["t"], res_x["y"], res_x["ref"], res_x["u"], u_max=10.0)
# k_x = compute_kpis(m_x)
# r_x = recommend_action(m_x, k_x)
# print_dashboard(m_x, k_x, r_x)

### Instructor Note

The automated recommendation is a starting point, not gospel. Encourage students to
challenge the algorithm's output. A student who says *'the algorithm says continue
running, but the saturation is at 28% and trending up — I'd inspect soon'* is showing
exactly the right thinking.